# PixelAI decoder training on Kaggle (1x or 2x T4)

**Notebook settings:** Accelerator = *GPU T4 x2*, Internet = *on*.

Run with **Save Version -> Save & Run All** so training continues after you close the browser.
Checkpoints go to `/kaggle/working/runs/...` and become this version's output.

**Resuming in a new session:** add the previous version's output as an input
(*Add Input -> Your Work*) and set `RESUME_FROM` below to its `runs/pixelai_decoder` folder.
Each teammate runs this under their own account; the commit hash is recorded in every checkpoint.

In [ ]:
import os
REPO_URL = "https://github.com/Youssef-Ahmed38/pixel-active-inference.git"
COMMIT = "main"                                                     # pin a commit hash for reported runs
DATA_DIR = "/tmp/pixelai"             # or a Kaggle Dataset path, e.g. /kaggle/input/pai-pixelai-data
OUT_DIR = "/kaggle/working/runs/pixelai_decoder"
RESUME_FROM = ""                      # e.g. /kaggle/input/<previous-version>/runs/pixelai_decoder
EXTRA = ""                            # config overrides, e.g. "decoder.image_size=256 train.batch_size=32"
os.environ["MUJOCO_GL"] = "egl"       # headless GPU rendering on Linux

In [ ]:
!git clone -q {REPO_URL} /kaggle/working/pai && git -C /kaggle/working/pai checkout -q {COMMIT}
%cd /kaggle/working/pai
!pip install -q -e .
!python scripts/fetch_assets.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Throughput numbers for the compute plan (a few minutes). Results in results/benchmark_<host>.json
!python scripts/benchmark.py

In [ ]:
# Collect training data here (skip if DATA_DIR points to an attached Kaggle Dataset).
if not DATA_DIR.startswith("/kaggle/input"):
    !python scripts/collect_data.py data.dir={DATA_DIR} data.workers=4 {EXTRA}

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
resume = f"--resume-from {RESUME_FROM}" if RESUME_FROM else ""
launcher = f"torchrun --nproc_per_node={n_gpu}" if n_gpu > 1 else "python"
!{launcher} scripts/train_decoder.py {resume} data.dir={DATA_DIR} train.out_dir={OUT_DIR} {EXTRA}

In [ ]:
from IPython.display import Image, display
display(Image(f"{OUT_DIR}/samples.png"))   # top: real renders, bottom: decoder predictions
!tail -n 3 {OUT_DIR}/log.jsonl